In [ ]:
# Retrivers is component in langchain that fetches relevant docu from data sourcs in responce to query.
# fucntion takes, quvery and give langchain document object
# All Retrivers are Runnable in Langchain

# Types of Retrivers
    # 1) Based on Data Source
        # a) Wikipedia retievrs
        # b) Youtbe retievrs
        # c) vector store retievers
        # d) Archive retrievers
        
    # 2) Based on search stragery
        # a) NMP 
        # b) Multiquery

In [1]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict , Annotated , List , Optional
import warnings
warnings.filterwarnings("ignore")
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser , JsonOutputParser 

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

parser = StrOutputParser()

llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash" , api_key= GOOGLE_API_KEY)
llm_gemini.invoke("who is father of india").content

c:\Users\singh\Let's Gooooo\Langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'Mahatma Gandhi is widely considered the Father of India.'

### Retrievers based on Data Type

In [7]:
# wikipedia retrievers
# query wikipedia api and retriever relevant articals
# perform searching so this is not document loder hence retriever

import chroma_db
from langchain_community.retrievers import WikipediaRetriever

retriever = WikipediaRetriever(top_k_results= 3 , lang= "en")

docs = retriever.invoke("Impact of WWI on indian Economy")

for i , doc  in enumerate(docs):
    print(f"for artical {i} title is {doc.page_content[:500]}")

for artical 0 title is World War I or the First World War (28 July 1914 – 11 November 1918), also known as the Great War, was a global conflict between two coalitions: the Allies (or Entente) and the Central Powers. Main areas of conflict included Europe and the Middle East, as well as parts of Africa and the Asia-Pacific. There were important developments in weaponry including tanks, aircraft, artillery, machine guns, and chemical weapons. One of the deadliest conflicts in history, it resulted in an estimated 30 mill
for artical 1 title is The Indian Armed Forces are the military forces of the Republic of India. It consists of three professional uniformed services: the Indian Army, the Indian Navy, and the Indian Air Force. Additionally, the Indian Armed Forces are supported by the Central Armed Police Forces, the Indian Coast Guard, and  the Special Frontier Force and various inter-service commands and institutions such as the Strategic Forces Command, the Andaman and Nicobar Command

In [10]:
# Vector Store Retrievers
# Most common, very famous
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

final_docu = [doc1 , doc2 , doc3 , doc4 , doc5]

# Embedding Model
embedding_model = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001')

vector_store = Chroma.from_documents(
    documents= final_docu,
    embedding= embedding_model,
    collection_name="my_collection"
)

In [ ]:
# Retrivers give us various staegory to seach as oppose to 
retriever = vector_store.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={'score_threshold': 0.5 , 'k' : 2}
        )

result = retriever.invoke("Most famous caption in indian team")

for i , doc  in enumerate(result):
    print(f"for artical {i+1} title is : {doc.page_content[:500]}")

for artical 1 title is : MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.
for artical 2 title is : Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.
